# Foundation Inventory Parts — Silver Layer

Reads current inventory part records from `lego_brickbase.bronze.raw_inventory_parts` and loads them into the `lego_brickbase.silver.foundation_inventory_parts` Delta table.

## What this notebook does

1. **Read** — Selects all current, non-deleted records from the bronze `raw_inventory_parts` table.
2. **Transform** — Retains all business columns (`inventory_id`, `part_num`, `color_id`, `quantity`, `is_spare`, `img_url`, `valid_from`) and derives a `valid_from_date` column used for partitioning. The composite key is `inventory_id` + `part_num` + `color_id`.
3. **Write** — Overwrites the Delta table at `/Volumes/lego_brickbase/silver/delta_volume/inventory_parts`, partitioned by `valid_from_date`.
4. **Register** — Creates the catalog table `lego_brickbase.silver.foundation_inventory_parts` in Unity Catalog if it does not already exist.

## Imports and Configuration

Import Spark functions needed for the transformation. `BRONZE_TABLE` is the source; `DELTA_TABLE_PATH` is the physical Delta location on the Unity Catalog external volume; `CATALOG_TABLE` is the Unity Catalog table reference.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_date

spark = SparkSession.builder.getOrCreate()

BRONZE_TABLE     = "lego_brickbase.bronze.raw_inventory_parts"
DELTA_TABLE_PATH = "/Volumes/lego_brickbase/silver/delta_volume/inventory_parts"
CATALOG_TABLE    = "lego_brickbase.silver.foundation_inventory_parts"

## Read and Transform

Filter the bronze table to current, non-deleted records, then select all business columns and derive `valid_from_date` (the date portion of `valid_from`) for use as the Delta partition column. The composite key `inventory_id` + `part_num` + `color_id` uniquely identifies each row.

In [ ]:
df_source = (
    spark.table(BRONZE_TABLE)
    .filter((col("is_current") == True) & (col("is_deleted") == False))
)

df_foundation = (
    df_source
    .select(
        col("inventory_id").alias("inventory_part_key"),
        col("part_num").alias("part_key"),
        col("color_id").alias("color_key"),
        col("part_num").alias("part_number"),
        col("inventory_id"),
        col("color_id"),
        col("quantity"),
        col("is_spare"),
        col("img_url").alias("image_url"),
        col("valid_from"),
    )
    .withColumn("valid_from_date", to_date(col("valid_from")))
)

df_foundation.printSchema()
df_foundation.display(10, truncate=False)

## Write to Delta Volume and Register Catalog Table

Overwrite the Delta table at the silver volume path, partitioned by `valid_from_date`. Then register it in Unity Catalog as `lego_brickbase.silver.foundation_inventory_parts` if it does not already exist.

In [ ]:
(
    df_foundation
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("valid_from_date")
    .save(DELTA_TABLE_PATH)
)

row_count = spark.read.format("delta").load(DELTA_TABLE_PATH).count()
print(f"Rows written to Delta table: {row_count:,}")

spark.sql(f"""
    CREATE OR REPLACE TABLE {CATALOG_TABLE}
    AS SELECT * FROM delta.`{DELTA_TABLE_PATH}`
""")
print(f"Catalog table ready: {CATALOG_TABLE}")